<a href="https://colab.research.google.com/github/Gokul120799/BRAIN-TUMOR-MRI-CLASSIFICATION/blob/main/Brain_Tumour.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
import pickle


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Define paths
train_dir = "/content/drive/MyDrive/Tumour/train"
val_dir   = "/content/drive/MyDrive/Tumour/valid"
test_dir  = "/content/drive/MyDrive/Tumour/test"

# Image size
IMG_SIZE = (224, 224)

# Data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(train_dir, target_size=IMG_SIZE, batch_size=32, class_mode="categorical")
val_gen   = val_datagen.flow_from_directory(val_dir, target_size=IMG_SIZE, batch_size=32, class_mode="categorical")
test_gen  = test_datagen.flow_from_directory(test_dir, target_size=IMG_SIZE, batch_size=32, class_mode="categorical")


Found 1695 images belonging to 4 classes.
Found 502 images belonging to 4 classes.
Found 246 images belonging to 4 classes.


In [ ]:
cnn_model = Sequential([
    Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)),
    MaxPooling2D(2,2),
    BatchNormalization(),

    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),
    BatchNormalization(),

    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),
    BatchNormalization(),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(train_gen.num_classes, activation="softmax")
])

cnn_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224,224,3))
for layer in base_model.layers:
    layer.trainable = False  # freeze base layers

x = Flatten()(base_model.output)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(train_gen.num_classes, activation="softmax")(x)

resnet_model = Model(inputs=base_model.input, outputs=output)
resnet_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("best_model.h5", save_best_only=True)
]

history_cnn = cnn_model.fit(train_gen, validation_data=val_gen, epochs=20, callbacks=callbacks)
history_resnet = resnet_model.fit(train_gen, validation_data=val_gen, epochs=20, callbacks=callbacks)


Epoch 1/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.4672 - loss: 6.2595

53/53 ━━━━━━━━━━━━━━━━━━━━ 677s 13s/step - accuracy: 0.4395 - loss: 3.9090 - val_accuracy: 0.2610 - val_loss: 2.4070
Epoch 2/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 233s 4s/step - accuracy: 0.4059 - loss: 1.5197 - val_accuracy: 0.2351 - val_loss: 3.4266
Epoch 3/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 264s 4s/step - accuracy: 0.4442 - loss: 1.6277 - val_accuracy: 0.2371 - val_loss: 3.3095
Epoch 4/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 229s 4s/step - accuracy: 0.4726 - loss: 1.4068 - val_accuracy: 0.2351 - val_loss: 15.0818
Epoch 5/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 233s 4s/step - accuracy: 0.4743 - loss: 1.2425 - val_accuracy: 0.3845 - val_loss: 2.8167
Epoch 6/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.4797 - loss: 1.2964

53/53 ━━━━━━━━━━━━━━━━━━━━ 230s 4s/step - accuracy: 0.4655 - loss: 1.3305 - val_accuracy: 0.3048 - val_loss: 1.5170
Epoch 7/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.4562 - loss: 1.2468

53/53 ━━━━━━━━━━━━━━━━━━━━ 241s 5s/step - accuracy: 0.4643 - loss: 1.2008 - val_accuracy: 0.5359 - val_loss: 1.3710
Epoch 8/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.4824 - loss: 1.2116

53/53 ━━━━━━━━━━━━━━━━━━━━ 232s 4s/step - accuracy: 0.4791 - loss: 1.2396 - val_accuracy: 0.5936 - val_loss: 1.1375
Epoch 9/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 231s 4s/step - accuracy: 0.4726 - loss: 1.1933 - val_accuracy: 0.6116 - val_loss: 1.4257
Epoch 10/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.5072 - loss: 1.3383

53/53 ━━━━━━━━━━━━━━━━━━━━ 242s 5s/step - accuracy: 0.4950 - loss: 1.2803 - val_accuracy: 0.5657 - val_loss: 1.1204
Epoch 11/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.4698 - loss: 1.2757

53/53 ━━━━━━━━━━━━━━━━━━━━ 237s 4s/step - accuracy: 0.4920 - loss: 1.2425 - val_accuracy: 0.6394 - val_loss: 1.0470
Epoch 12/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 231s 4s/step - accuracy: 0.4879 - loss: 1.2734 - val_accuracy: 0.6614 - val_loss: 1.1174
Epoch 13/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 245s 5s/step - accuracy: 0.4867 - loss: 1.2044 - val_accuracy: 0.5020 - val_loss: 1.2498
Epoch 14/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 232s 4s/step - accuracy: 0.4997 - loss: 1.2026 - val_accuracy: 0.5299 - val_loss: 1.1518
Epoch 15/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.4844 - loss: 1.1481

53/53 ━━━━━━━━━━━━━━━━━━━━ 242s 5s/step - accuracy: 0.5015 - loss: 1.1201 - val_accuracy: 0.6554 - val_loss: 0.9293
Epoch 16/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.4919 - loss: 1.1602

53/53 ━━━━━━━━━━━━━━━━━━━━ 233s 4s/step - accuracy: 0.4938 - loss: 1.1237 - val_accuracy: 0.6713 - val_loss: 0.9253
Epoch 17/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 236s 4s/step - accuracy: 0.4973 - loss: 1.1180 - val_accuracy: 0.5637 - val_loss: 1.0747
Epoch 18/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 239s 5s/step - accuracy: 0.4950 - loss: 1.1354 - val_accuracy: 0.6494 - val_loss: 0.9787
Epoch 19/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 234s 4s/step - accuracy: 0.5097 - loss: 1.1984 - val_accuracy: 0.6514 - val_loss: 0.9526
Epoch 20/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 241s 5s/step - accuracy: 0.4850 - loss: 1.1870 - val_accuracy: 0.4661 - val_loss: 5.5894
Epoch 1/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 403s 7s/step - accuracy: 0.3381 - loss: 3.2369 - val_accuracy: 0.3845 - val_loss: 1.3733
Epoch 2/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 400s 8s/step - accuracy: 0.3617 - loss: 1.3776 - val_accuracy: 0.4263 - val_loss: 1.3180
Epoch 3/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 451s 9s/step - accuracy: 0.3935 - loss: 1.3221 - val_accuracy: 0.4183 - val_loss

In [ ]:
# Evaluate CNN
cnn_preds = cnn_model.predict(test_gen)
print(classification_report(test_gen.classes, np.argmax(cnn_preds, axis=1)))

# Evaluate ResNet
resnet_preds = resnet_model.predict(test_gen)
print(classification_report(test_gen.classes, np.argmax(resnet_preds, axis=1)))


8/8 ━━━━━━━━━━━━━━━━━━━━ 64s 9s/step
              precision    recall  f1-score   support

           0       0.33      0.44      0.38        80
           1       0.42      0.08      0.13        63
           2       0.27      0.22      0.24        49
           3       0.24      0.39      0.30        54

    accuracy                           0.29       246
   macro avg       0.31      0.28      0.26       246
weighted avg       0.32      0.29      0.27       246

8/8 ━━━━━━━━━━━━━━━━━━━━ 46s 5s/step
              precision    recall  f1-score   support

           0       0.33      0.93      0.49        80
           1       0.00      0.00      0.00        63
           2       0.26      0.12      0.17        49
           3       0.00      0.00      0.00        54

    accuracy                           0.33       246
   macro avg       0.15      0.26      0.16       246
weighted avg       0.16      0.33      0.19       246



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import os
import pickle

# -------------------------------
# Save CNN model (overwrite if exists)
# -------------------------------
cnn_model_path = "cnn_model.h5"
if os.path.exists(cnn_model_path):
    os.replace(cnn_model_path, cnn_model_path)  # ensures overwrite
cnn_model.save(cnn_model_path)

# -------------------------------
# Save ResNet model (overwrite if exists)
# -------------------------------
resnet_model_path = "resnet_model.h5"
if os.path.exists(resnet_model_path):
    os.replace(resnet_model_path, resnet_model_path)
resnet_model.save(resnet_model_path)

# -------------------------------
# Save class indices (overwrite if exists)
# -------------------------------
class_indices_path = "class_indices.pkl"
if os.path.exists(class_indices_path):
    os.replace(class_indices_path, class_indices_path)

with open(class_indices_path, "wb") as f:
    pickle.dump(train_gen.class_indices, f)


In [ ]:
!pip install streamlit
!pip install pyngrok


In [ ]:
import streamlit as st
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import pickle

# -------------------------------
# Load models and class indices
# -------------------------------
cnn_model = load_model("cnn_model.h5")
resnet_model = load_model("resnet_model.h5")

with open("class_indices.pkl", "rb") as f:
    class_indices = pickle.load(f)

# Reverse mapping: index → class name
class_labels = {v: k for k, v in class_indices.items()}

# -------------------------------
# Helper function: preprocess image
# -------------------------------
def preprocess_image(img_file):
    img = image.load_img(img_file, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    return img_array

# -------------------------------
# Streamlit UI
# -------------------------------
st.title("🧠 Brain Tumor MRI Classification")
st.write("Upload an MRI image to classify tumor type using trained models.")

uploaded_file = st.file_uploader("Upload MRI Image", type=["jpg", "jpeg", "png"])

model_choice = st.selectbox("Choose Model", ["Custom CNN", "ResNet50 Transfer Learning"])

if uploaded_file is not None:
    st.image(uploaded_file, caption="Uploaded MRI", use_column_width=True)

    if st.button("Predict"):
        img_array = preprocess_image(uploaded_file)

        if model_choice == "Custom CNN":
            preds = cnn_model.predict(img_array)
        else:
            preds = resnet_model.predict(img_array)

        pred_class = np.argmax(preds, axis=1)[0]
        confidence = np.max(preds)

        st.success(f"Predicted Tumor Type: **{class_labels[pred_class]}**")
        st.info(f"Confidence Score: {confidence:.2f}")


2026-08-23 15:34:57.648 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-23 15:34:57.926 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-08-23 15:34:57.930 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-23 15:34:57.933 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-23 15:34:57.944 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-23 15:34:57.947 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-23 15:34:57.951 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-23 15:34:57.953 Thread 'MainThread': mi

In [ ]:
!pip install streamlit pyngrok


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 51.8 MB/s eta 0:00:00


In [ ]:
# Save Streamlit app into app.py
app_code = """
import streamlit as st
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import pickle
import os

# -------------------------------
# Base directory for models/files
# -------------------------------
BASE_DIR = "/content"

# -------------------------------
# Safe model loader
# -------------------------------
def safe_load_model(filename):
    path = os.path.join(BASE_DIR, filename)
    try:
        model = load_model(path, compile=False)
        model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
        return model
    except Exception as e:
        st.error(f"Error loading {filename}: {e}")
        return None

# Load models
cnn_model = safe_load_model("cnn_model.h5")
resnet_model = safe_load_model("resnet_model.h5")
best_model = safe_load_model("best_model.h5")

# -------------------------------
# Load class indices
# -------------------------------
try:
    with open(os.path.join(BASE_DIR, "class_indices.pkl"), "rb") as f:
        class_indices = pickle.load(f)
    class_labels = {v: k for k, v in class_indices.items()}
except Exception as e:
    st.error(f"Error loading class indices: {e}")
    class_labels = {}

# -------------------------------
# Helper function: preprocess image
# -------------------------------
def preprocess_image(img_file):
    img = image.load_img(img_file, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    return img_array

# -------------------------------
# Prediction function
# -------------------------------
def predict_tumor(model, img_array):
    if model is None:
        return "Model not available", 0.0
    preds = model.predict(img_array)
    pred_class = np.argmax(preds, axis=1)[0]
    confidence = np.max(preds)
    return class_labels.get(pred_class, "Unknown"), confidence

# -------------------------------
# Streamlit UI
# -------------------------------
st.title("🧠 Brain Tumor MRI Classification")
st.write("Upload an MRI image to classify tumor type using trained models.")

uploaded_file = st.file_uploader("Upload MRI Image", type=["jpg", "jpeg", "png"])
model_choice = st.selectbox("Choose Model", ["Custom CNN", "ResNet50 Transfer Learning", "Best Model"])

if uploaded_file is not None:
    st.image(uploaded_file, caption="Uploaded MRI", use_column_width=True)

if st.button("Predict"):
    if uploaded_file is not None:
        img_array = preprocess_image(uploaded_file)

        if model_choice == "Custom CNN":
            tumor_type, confidence = predict_tumor(cnn_model, img_array)
        elif model_choice == "ResNet50 Transfer Learning":
            tumor_type, confidence = predict_tumor(resnet_model, img_array)
        else:
            tumor_type, confidence = predict_tumor(best_model, img_array)

        st.success(f"Predicted Tumor Type: **{tumor_type}**")
        st.info(f"Confidence Score: {confidence:.2f}")
    else:
        st.warning("Please upload an MRI image first.")
"""

# Write to file
with open("app.py", "w") as f:
    f.write(app_code)


In [ ]:
!pwd


/content


In [ ]:
!pip install streamlit pyngrok
!npm install -g localtunnel
!streamlit run app.py & npx localtunnel --port 8501


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
changed 22 packages in 4s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙⠙

⠹⠸⠼⠴⠦⠧your url is: https://smart-worms-kneel.loca.lt
2026-08-24 03:38:37.466 Uvicorn server started on :::8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://104.155.194.36:8502

  Stopping...
^C


In [ ]:
!pkill -f streamlit || true
!pkill -f ngrok || true

print("✅ Old Streamlit/ngrok processes stopped.")

^C
^C
✅ Old Streamlit/ngrok processes stopped.


In [ ]:
import os
import time
import subprocess

APP_PATH = "/content/app.py"
LOG_PATH = "/content/streamlit.log"

# ============================================================
# 1. CHECK APP
# ============================================================

if not os.path.exists(APP_PATH):
    raise FileNotFoundError(
        f"❌ app.py not found at {APP_PATH}"
    )

print("✅ app.py found:", APP_PATH)


# ============================================================
# 2. KILL ANY PROCESS USING PORT 8501
# ============================================================

print("\n🧹 Cleaning port 8501...")

os.system("fuser -k 8501/tcp >/dev/null 2>&1 || true")

# Also stop old Streamlit processes
os.system("pkill -9 -f 'streamlit run' >/dev/null 2>&1 || true")

time.sleep(3)

print("✅ Port 8501 cleaned.")


# ============================================================
# 3. START STREAMLIT
# ============================================================

print("\n🚀 Starting Streamlit...")

log_file = open(
    LOG_PATH,
    "w"
)

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        APP_PATH,
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--browser.gatherUsageStats=false"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print("PID:", process.pid)

# Wait for Streamlit
time.sleep(8)


# ============================================================
# 4. CHECK PROCESS
# ============================================================

if process.poll() is None:

    print("\n✅ Streamlit process is running!")
    print("🌐 Port: 8501")

else:

    print("\n❌ Streamlit stopped unexpectedly.")

    log_file.close()

    print("\n========== STREAMLIT LOG ==========\n")

    with open(LOG_PATH, "r") as f:
        print(f.read())

✅ app.py found: /content/app.py

🧹 Cleaning port 8501...
✅ Port 8501 cleaned.

🚀 Starting Streamlit...
PID: 21316

✅ Streamlit process is running!
🌐 Port: 8501


In [ ]:
import requests

try:

    response = requests.get(
        "http://127.0.0.1:8501",
        timeout=10
    )

    print("✅ Streamlit is responding!")
    print("Status code:", response.status_code)

except Exception as e:

    print("❌ Streamlit is not responding.")
    print("Error:", e)

    print("\n========== LOG ==========\n")

    with open("/content/streamlit.log", "r") as f:
        print(f.read())

✅ Streamlit is responding!
Status code: 200


In [ ]:
from pyngrok import ngrok

# Stop existing tunnels
try:
    ngrok.kill()
except:
    pass

# ------------------------------------------------------------
# ENTER YOUR NGROK TOKEN
# ------------------------------------------------------------

NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Create public tunnel
tunnel = ngrok.connect(
    8501,
    "http"
)

public_url = tunnel.public_url

print("=" * 60)
print("🧠 BRAIN TUMOR MRI CLASSIFICATION")
print("=" * 60)
print()
print("✅ Streamlit is running")
print("✅ Ngrok tunnel created")
print()
print("🔗 OPEN YOUR APP:")
print()
print(public_url)
print()
print("=" * 60)

ERROR:pyngrok.process.ngrok:t=2026-08-24T04:11:14+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_AUTH_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-08-24T04:11:14+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_AUTH_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_AUTH_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.

In [ ]:
from pyngrok import ngrok

# ============================================================
# NGROK AUTHENTICATION
# ============================================================

NGROK_AUTH_TOKEN = "3ILXIYhycZM7L6kGLbk2Pf3f4Xa_6axhLsCFDxSKPCz3Q1cFB"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("✅ Ngrok authentication configured.")

✅ Ngrok authentication configured.


In [ ]:
from pyngrok import ngrok

# Close any previous tunnels
try:
    ngrok.kill()
except Exception:
    pass

# Create tunnel to Streamlit
tunnel = ngrok.connect(
    addr=8501,
    proto="http"
)

public_url = tunnel.public_url

print("=" * 60)
print("🧠 BRAIN TUMOR MRI CLASSIFICATION")
print("=" * 60)
print()
print("✅ Streamlit is running")
print("✅ Ngrok tunnel is connected")
print()
print("🔗 YOUR STREAMLIT APP:")
print()
print(public_url)
print()
print("=" * 60)

🧠 BRAIN TUMOR MRI CLASSIFICATION

✅ Streamlit is running
✅ Ngrok tunnel is connected

🔗 YOUR STREAMLIT APP:

https://winking-ignore-upstage.ngrok-free.dev



In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount("/content/drive")

# Main project folder
PROJECT_DIR = "/content/drive/MyDrive/Brain_Tumor_Project"

# Create project folder
os.makedirs(PROJECT_DIR, exist_ok=True)

print("✅ Google Drive mounted")
print("📁 Project folder:")
print(PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted
📁 Project folder:
/content/drive/MyDrive/Brain_Tumor_Project


In [ ]:
import os

print("=" * 60)
print("🔍 CHECKING BRAIN TUMOR PROJECT FILES")
print("=" * 60)

files_to_check = [
    "/content/app.py",
    "/content/cnn_model.h5",
    "/content/resnet_model.h5",
    "/content/best_model.h5",
    "/content/class_indices.pkl"
]

all_files_available = True

for file_path in files_to_check:

    if os.path.exists(file_path):

        size_mb = os.path.getsize(file_path) / (1024 * 1024)

        print(
            f"✅ {os.path.basename(file_path):25s}"
            f" {size_mb:.2f} MB"
        )

    else:

        print(
            f"❌ MISSING: {file_path}"
        )

        all_files_available = False

print("=" * 60)

if all_files_available:
    print("✅ ALL REQUIRED FILES ARE AVAILABLE")
else:
    print("⚠️ SOME FILES ARE MISSING")

🔍 CHECKING BRAIN TUMOR PROJECT FILES
✅ app.py                    0.01 MB
✅ cnn_model.h5              127.90 MB
✅ resnet_model.h5           237.55 MB
✅ best_model.h5             127.90 MB
✅ class_indices.pkl         0.00 MB
✅ ALL REQUIRED FILES ARE AVAILABLE


In [ ]:
import os
import shutil

PROJECT_DIR = "/content/drive/MyDrive/Brain_Tumor_Project"

# Files to save
files_to_save = [
    "app.py",
    "cnn_model.h5",
    "resnet_model.h5",
    "best_model.h5",
    "class_indices.pkl"
]

print("=" * 60)
print("💾 SAVING BRAIN TUMOR PROJECT")
print("=" * 60)

for filename in files_to_save:

    source = os.path.join(
        "/content",
        filename
    )

    destination = os.path.join(
        PROJECT_DIR,
        filename
    )

    if os.path.exists(source):

        shutil.copy2(
            source,
            destination
        )

        size_mb = os.path.getsize(source) / (1024 * 1024)

        print(
            f"✅ Saved: {filename}"
            f" ({size_mb:.2f} MB)"
        )

    else:

        print(
            f"❌ NOT FOUND: {source}"
        )

print("=" * 60)
print("✅ PROJECT FILES SAVED")
print("📁 Location:")
print(PROJECT_DIR)

💾 SAVING BRAIN TUMOR PROJECT
✅ Saved: app.py (0.01 MB)
✅ Saved: cnn_model.h5 (127.90 MB)
✅ Saved: resnet_model.h5 (237.55 MB)
✅ Saved: best_model.h5 (127.90 MB)
✅ Saved: class_indices.pkl (0.00 MB)
✅ PROJECT FILES SAVED
📁 Location:
/content/drive/MyDrive/Brain_Tumor_Project


In [ ]:
requirements = """streamlit
pyngrok
tensorflow
numpy
pillow
"""

requirements_path = os.path.join(
    PROJECT_DIR,
    "requirements.txt"
)

with open(
    requirements_path,
    "w"
) as f:

    f.write(requirements)

print("✅ requirements.txt created")
print(requirements_path)

✅ requirements.txt created
/content/drive/MyDrive/Brain_Tumor_Project/requirements.txt


In [ ]:
restore_code = r'''
import os
import shutil

# ============================================================
# BRAIN TUMOR PROJECT RESTORE SCRIPT
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/Brain_Tumor_Project"

FILES = [
    "app.py",
    "cnn_model.h5",
    "resnet_model.h5",
    "best_model.h5",
    "class_indices.pkl"
]

print("=" * 60)
print("🔄 RESTORING BRAIN TUMOR PROJECT")
print("=" * 60)

for filename in FILES:

    source = os.path.join(
        PROJECT_DIR,
        filename
    )

    destination = os.path.join(
        "/content",
        filename
    )

    if os.path.exists(source):

        shutil.copy2(
            source,
            destination
        )

        print(
            f"✅ Restored: {filename}"
        )

    else:

        print(
            f"❌ Missing from Drive: {filename}"
        )

print("=" * 60)
print("✅ RESTORE COMPLETE")
print("=" * 60)
'''

restore_path = os.path.join(
    PROJECT_DIR,
    "restore_project.py"
)

with open(
    restore_path,
    "w"
) as f:

    f.write(restore_code)

print("✅ Restore script created")
print(restore_path)

✅ Restore script created
/content/drive/MyDrive/Brain_Tumor_Project/restore_project.py


In [ ]:
import shutil
import os
from datetime import datetime

PROJECT_DIR = "/content/drive/MyDrive/Brain_Tumor_Project"

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

zip_base = f"/content/Brain_Tumor_Project_Backup_{timestamp}"

zip_file = shutil.make_archive(
    zip_base,
    "zip",
    PROJECT_DIR
)

print("=" * 60)
print("📦 BACKUP CREATED")
print("=" * 60)
print("✅ ZIP file:")
print(zip_file)

size_mb = os.path.getsize(zip_file) / (1024 * 1024)

print(f"📦 Size: {size_mb:.2f} MB")

# Copy ZIP to Google Drive
drive_zip = os.path.join(
    PROJECT_DIR,
    os.path.basename(zip_file)
)

shutil.copy2(
    zip_file,
    drive_zip
)

print()
print("✅ ZIP backup copied to Google Drive:")
print(drive_zip)

✅ Streamlit app created successfully.
📁 App path: /content/app.py
✅ app.py exists.
📦 Size: 8708 bytes


In [23]:
import shutil
import os
from datetime import datetime

PROJECT_DIR = "/content/drive/MyDrive/Brain_Tumor_Project"

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

zip_base = f"/content/Brain_Tumor_Project_Backup_{timestamp}"

zip_file = shutil.make_archive(
    zip_base,
    "zip",
    PROJECT_DIR
)

print("=" * 60)
print("📦 BACKUP CREATED")
print("=" * 60)
print("✅ ZIP file:")
print(zip_file)

size_mb = os.path.getsize(zip_file) / (1024 * 1024)

print(f"📦 Size: {size_mb:.2f} MB")

# Copy ZIP to Google Drive
drive_zip = os.path.join(
    PROJECT_DIR,
    os.path.basename(zip_file)
)

shutil.copy2(
    zip_file,
    drive_zip
)

print()
print("✅ ZIP backup copied to Google Drive:")
print(drive_zip)

📦 BACKUP CREATED
✅ ZIP file:
/content/Brain_Tumor_Project_Backup_20260824_043655.zip
📦 Size: 754.07 MB

✅ ZIP backup copied to Google Drive:
/content/drive/MyDrive/Brain_Tumor_Project/Brain_Tumor_Project_Backup_20260824_043655.zip


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Brain_Tumor_Project"

files = [
    "app.py",
    "cnn_model.h5",
    "resnet_model.h5",
    "best_model.h5",
    "class_indices.pkl",
    "requirements.txt",
    "restore_project.py"
]

print("=" * 70)
print("🔍 FINAL PROJECT VERIFICATION")
print("=" * 70)

all_good = True

for filename in files:

    path = os.path.join(
        PROJECT_DIR,
        filename
    )

    if os.path.exists(path):

        size_mb = os.path.getsize(path) / (1024 * 1024)

        print(
            f"✅ {filename:25s} "
            f"{size_mb:.2f} MB"
        )

    else:

        print(
            f"❌ {filename:25s} MISSING"
        )

        all_good = False

print("=" * 70)

if all_good:

    print("🎉 COMPLETE PROJECT SUCCESSFULLY SAVED!")
    print()
    print("📁 Google Drive:")
    print(PROJECT_DIR)

else:

    print("⚠️ Some files are missing.")

🔍 FINAL PROJECT VERIFICATION
✅ app.py                    0.01 MB
✅ cnn_model.h5              127.90 MB
✅ resnet_model.h5           237.55 MB
✅ best_model.h5             127.90 MB
✅ class_indices.pkl         0.00 MB
✅ requirements.txt          0.00 MB
✅ restore_project.py        0.00 MB
🎉 COMPLETE PROJECT SUCCESSFULLY SAVED!

📁 Google Drive:
/content/drive/MyDrive/Brain_Tumor_Project


In [22]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Brain_Tumor_Project"

files = [
    "app.py",
    "cnn_model.h5",
    "resnet_model.h5",
    "best_model.h5",
    "class_indices.pkl",
    "requirements.txt",
    "restore_project.py"
]

print("=" * 70)
print("🔍 FINAL PROJECT VERIFICATION")
print("=" * 70)

all_good = True

for filename in files:

    path = os.path.join(
        PROJECT_DIR,
        filename
    )

    if os.path.exists(path):

        size_mb = os.path.getsize(path) / (1024 * 1024)

        print(
            f"✅ {filename:25s} "
            f"{size_mb:.2f} MB"
        )

    else:

        print(
            f"❌ {filename:25s} MISSING"
        )

        all_good = False

print("=" * 70)

if all_good:

    print("🎉 COMPLETE PROJECT SUCCESSFULLY SAVED!")
    print()
    print("📁 Google Drive:")
    print(PROJECT_DIR)

else:

    print("⚠️ Some files are missing.")

🔍 FINAL PROJECT VERIFICATION
✅ app.py                    0.01 MB
✅ cnn_model.h5              127.90 MB
✅ resnet_model.h5           237.55 MB
✅ best_model.h5             127.90 MB
✅ class_indices.pkl         0.00 MB
✅ requirements.txt          0.00 MB
✅ restore_project.py        0.00 MB
🎉 COMPLETE PROJECT SUCCESSFULLY SAVED!

📁 Google Drive:
/content/drive/MyDrive/Brain_Tumor_Project


In [24]:
# ============================================================
# 🧠 BRAIN TUMOR MRI CLASSIFICATION
# ONE-CELL REUSABLE COLAB LAUNCHER
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================

import os
import shutil
import subprocess
import time
import getpass
import requests

print("=" * 70)
print("🧠 BRAIN TUMOR MRI CLASSIFICATION")
print("🚀 ONE-CELL REUSABLE COLAB LAUNCHER")
print("=" * 70)


# ============================================================
# 2. MOUNT GOOGLE DRIVE
# ============================================================

print("\n📂 Mounting Google Drive...")

from google.colab import drive

try:
    drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully.")

except Exception as e:
    print("❌ Google Drive mounting failed.")
    print("Error:", e)
    raise


# ============================================================
# 3. PROJECT LOCATION
# ============================================================

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "Brain_Tumor_Project"
)

print("\n📁 Project directory:")
print(PROJECT_DIR)


# ============================================================
# 4. CHECK PROJECT DIRECTORY
# ============================================================

if not os.path.exists(PROJECT_DIR):

    raise FileNotFoundError(
        "\n❌ Brain_Tumor_Project folder was not found.\n\n"
        "Expected location:\n"
        f"{PROJECT_DIR}\n\n"
        "Please make sure your project has been saved "
        "to Google Drive."
    )

print("✅ Project folder found.")


# ============================================================
# 5. REQUIRED FILES
# ============================================================

REQUIRED_FILES = [
    "app.py",
    "cnn_model.h5",
    "resnet_model.h5",
    "best_model.h5",
    "class_indices.pkl"
]


# ============================================================
# 6. RESTORE FILES TO /content
# ============================================================

print("\n" + "=" * 70)
print("🔄 RESTORING PROJECT FILES")
print("=" * 70)

restore_errors = []

for filename in REQUIRED_FILES:

    source = os.path.join(
        PROJECT_DIR,
        filename
    )

    destination = os.path.join(
        "/content",
        filename
    )

    if os.path.exists(source):

        try:

            shutil.copy2(
                source,
                destination
            )

            size_mb = (
                os.path.getsize(source)
                / (1024 * 1024)
            )

            print(
                f"✅ {filename:25s}"
                f" {size_mb:.2f} MB"
            )

        except Exception as e:

            print(
                f"❌ Failed: {filename}"
            )

            print(
                f"   Error: {e}"
            )

            restore_errors.append(
                filename
            )

    else:

        print(
            f"❌ Missing from Drive: {filename}"
        )

        restore_errors.append(
            filename
        )


# ============================================================
# 7. STOP IF FILES ARE MISSING
# ============================================================

if restore_errors:

    print("\n❌ PROJECT RESTORE FAILED")

    print(
        "\nMissing/failed files:"
    )

    for filename in restore_errors:
        print(
            f"   - {filename}"
        )

    raise FileNotFoundError(
        "\nPlease check your Google Drive project folder."
    )

print("\n✅ All project files restored.")


# ============================================================
# 8. VERIFY /content FILES
# ============================================================

print("\n" + "=" * 70)
print("🔍 VERIFYING RESTORED FILES")
print("=" * 70)

for filename in REQUIRED_FILES:

    path = os.path.join(
        "/content",
        filename
    )

    if os.path.exists(path):

        print(
            f"✅ /content/{filename}"
        )

    else:

        raise FileNotFoundError(
            f"❌ File not found: {path}"
        )


# ============================================================
# 9. INSTALL REQUIRED PACKAGES
# ============================================================

print("\n" + "=" * 70)
print("📦 INSTALLING REQUIRED PACKAGES")
print("=" * 70)

subprocess.run(
    [
        "pip",
        "install",
        "-q",
        "streamlit",
        "pyngrok",
        "tensorflow",
        "pillow",
        "numpy"
    ],
    check=True
)

print("✅ Required packages installed.")


# ============================================================
# 10. STOP OLD STREAMLIT PROCESS
# ============================================================

print("\n" + "=" * 70)
print("🧹 CLEANING PORT 8501")
print("=" * 70)

# Kill anything currently using port 8501
os.system(
    "fuser -k 8501/tcp "
    "> /dev/null 2>&1 || true"
)

# Kill previous Streamlit processes
os.system(
    "pkill -9 -f 'streamlit run' "
    "> /dev/null 2>&1 || true"
)

# Give processes time to terminate
time.sleep(3)

print("✅ Port 8501 cleaned.")


# ============================================================
# 11. START STREAMLIT
# ============================================================

print("\n" + "=" * 70)
print("🚀 STARTING STREAMLIT")
print("=" * 70)

APP_PATH = "/content/app.py"

LOG_PATH = "/content/streamlit.log"

log_file = open(
    LOG_PATH,
    "w"
)

streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        APP_PATH,

        "--server.port=8501",

        "--server.address=0.0.0.0",

        "--server.headless=true",

        "--browser.gatherUsageStats=false"
    ],

    stdout=log_file,

    stderr=subprocess.STDOUT
)

print(
    "Streamlit PID:",
    streamlit_process.pid
)

# Wait for startup
time.sleep(8)


# ============================================================
# 12. CHECK STREAMLIT PROCESS
# ============================================================

if streamlit_process.poll() is not None:

    log_file.close()

    print("\n❌ STREAMLIT FAILED TO START")

    print(
        "\n========== STREAMLIT LOG ==========\n"
    )

    with open(
        LOG_PATH,
        "r"
    ) as f:

        print(
            f.read()
        )

    raise RuntimeError(
        "Streamlit process stopped."
    )


print(
    "✅ Streamlit process is running."
)


# ============================================================
# 13. CHECK PORT 8501
# ============================================================

print(
    "\n🔍 Checking Streamlit server..."
)

streamlit_ready = False

for attempt in range(10):

    try:

        response = requests.get(
            "http://127.0.0.1:8501",
            timeout=3
        )

        if response.status_code == 200:

            streamlit_ready = True

            break

    except Exception:

        pass

    time.sleep(2)


if not streamlit_ready:

    log_file.close()

    print(
        "\n❌ Streamlit is not responding."
    )

    print(
        "\n========== STREAMLIT LOG ==========\n"
    )

    with open(
        LOG_PATH,
        "r"
    ) as f:

        print(
            f.read()
        )

    raise RuntimeError(
        "Streamlit server is not responding."
    )


print(
    "✅ Streamlit is responding on port 8501."
)


# ============================================================
# 14. NGROK
# ============================================================

print("\n" + "=" * 70)
print("🌐 NGROK CONFIGURATION")
print("=" * 70)

from pyngrok import ngrok


# ------------------------------------------------------------
# ENTER TOKEN SECURELY
# ------------------------------------------------------------

print(
    "\n🔐 Enter your ngrok authtoken."
)

print(
    "Your token will NOT be displayed."
)

NGROK_AUTH_TOKEN = getpass.getpass(
    "Ngrok authtoken: "
)


# ============================================================
# 15. VALIDATE TOKEN
# ============================================================

if not NGROK_AUTH_TOKEN.strip():

    raise ValueError(
        "❌ Ngrok token was empty."
    )


# ============================================================
# 16. CONFIGURE NGROK
# ============================================================

try:

    ngrok.set_auth_token(
        NGROK_AUTH_TOKEN.strip()
    )

    print(
        "✅ Ngrok authentication configured."
    )

except Exception as e:

    print(
        "❌ Ngrok authentication failed."
    )

    print(
        "Error:",
        e
    )

    raise


# ============================================================
# 17. CLOSE OLD NGROK TUNNELS
# ============================================================

try:

    ngrok.kill()

except Exception:

    pass


time.sleep(2)


# ============================================================
# 18. CREATE NGROK TUNNEL
# ============================================================

print(
    "\n🚀 Creating ngrok tunnel..."
)

try:

    tunnel = ngrok.connect(
        addr=8501,
        proto="http"
    )

except Exception as e:

    print(
        "\n❌ Could not create ngrok tunnel."
    )

    print(
        "Error:",
        e
    )

    raise


# ============================================================
# 19. GET PUBLIC URL
# ============================================================

public_url = tunnel.public_url


# ============================================================
# 20. FINAL RESULT
# ============================================================

print("\n")

print("=" * 70)
print("🎉 BRAIN TUMOR MRI CLASSIFICATION IS LIVE")
print("=" * 70)

print()

print("✅ Google Drive        : Connected")
print("✅ Project files       : Restored")
print("✅ Models              : Available")
print("✅ Streamlit           : Running")
print("✅ Port                : 8501")
print("✅ Ngrok               : Connected")

print()

print("🔗 YOUR STREAMLIT APP:")
print()

print(public_url)

print()

print("=" * 70)
print("👉 OPEN THE URL ABOVE IN YOUR BROWSER")
print("=" * 70)

print()

print(
    "⚠️ Keep this Colab runtime running while "
    "you use the application."
)

🧠 BRAIN TUMOR MRI CLASSIFICATION
🚀 ONE-CELL REUSABLE COLAB LAUNCHER

📂 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully.

📁 Project directory:
/content/drive/MyDrive/Brain_Tumor_Project
✅ Project folder found.

🔄 RESTORING PROJECT FILES
✅ app.py                    0.01 MB
✅ cnn_model.h5              127.90 MB
✅ resnet_model.h5           237.55 MB
✅ best_model.h5             127.90 MB
✅ class_indices.pkl         0.00 MB

✅ All project files restored.

🔍 VERIFYING RESTORED FILES
✅ /content/app.py
✅ /content/cnn_model.h5
✅ /content/resnet_model.h5
✅ /content/best_model.h5
✅ /content/class_indices.pkl

📦 INSTALLING REQUIRED PACKAGES
✅ Required packages installed.

🧹 CLEANING PORT 8501
✅ Port 8501 cleaned.

🚀 STARTING STREAMLIT
Streamlit PID: 28338
✅ Streamlit process is running.

🔍 Checking Streamlit server...
✅ Streamlit is responding on port 8501.

